In [ ]:
import os, re
import numpy as np, pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt, seaborn as sns
from datetime import datetime
from sklearn.model_selection import StratifiedGroupKFold, ParameterGrid
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support, classification_report, confusion_matrix
)

In [ ]:
base_dir = "./Aggregated"
AGGREGATED_LABELS = os.path.join(base_dir, "AllTales-Aggregated.csv")

SENTIMENT_COL = "Sentiments-Aggregated"
MULTI_FLAG_COL = "Multi-Aggregated"
JOIN_KEYS = ["Story", "Segment"]

SENTIMENTS = ["Negative", "Neutral", "Positive"]

USE_GPU = True
EARLY_STOP_ROUNDS = 150
N_ESTIMATORS_MAX = 2500
SAVE_CM_PLOTS = True
SMOOTH_WINDOW = 3
CB_BETA = 0.99
RANDOM_STATE = 42

PARAM_GRID = {
    "max_depth": [4, 5, 6],
    "min_child_weight": [1, 1.5, 2],
    "eta": [0.05, 0.06, 0.07],
    "gamma": [0.5, 0.1, 0.15, 0.2],
    "subsample": [0.85, 0.9, 0.95],
    "colsample_bytree": [0.75, 0.8, 0.85],
    "lambda": [2.0, 2.5, 3.0],
    "alpha": [0.4, 0.5, 0.6],
    "scale_pos_weight": [0.9, 1.0],
}


timestamp = datetime.now().strftime("%y%m%d-%H%M")
out_dir = os.path.join(base_dir, "results", f"{timestamp}-RSGKF-sentiment")
os.makedirs(out_dir, exist_ok=True)


In [ ]:
def class_balanced_weights(y, num_classes, beta=0.99):
    counts = np.bincount(y, minlength=num_classes).astype(float)
    eff_num = 1.0 - np.power(beta, counts)
    eff_num[eff_num == 0] = 1.0
    w = (1.0 - beta) / eff_num
    w *= num_classes / np.sum(w)
    return w

def build_sample_weights(y, class_w):
    return class_w[y]

def variance_filter_columns(X):
    return list(X.columns[X.var(axis=0) > 0.0])

def storywise_zscore_fit(df, cols):
    stats = {}
    for s, g in df.groupby("Story"):
        mu, sd = g[cols].mean(), g[cols].std().replace(0, 1)
        stats[s] = (mu, sd)
    return stats

def storywise_zscore_transform(df, cols, stats):
    out = df.copy()
    for s, idx in out.groupby("Story").groups.items():
        mu, sd = stats.get(s, (out[cols].mean(), out[cols].std().replace(0, 1)))
        out.loc[idx, cols] = (out.loc[idx, cols] - mu) / sd
    return out

def smooth_probs_per_story(df, y_proba, class_names, window=3):
    if y_proba.size == 0:
        return y_proba
    df_local = df.reset_index(drop=True)
    proba_df = pd.DataFrame(y_proba, columns=class_names, index=df_local.index)
    proba_df = proba_df.join(df_local[["Story"]])
    smoothed = []
    for _, g in proba_df.groupby("Story", sort=False):
        w = max(1, min(window, max(1, len(g)//6)))
        if len(g) < 20:
            roll = g[class_names]
        else:
            roll = g[class_names].rolling(w, center=True, min_periods=1).median()
        roll.index = g.index
        smoothed.append(roll)
    return pd.concat(smoothed).sort_index().values

def make_params_base_multiclass(num_class, use_gpu):
    params = dict(
        n_estimators=N_ESTIMATORS_MAX,
        n_jobs=-1,
        objective="multi:softprob",
        num_class=num_class,
        eta=0.05,
        max_depth=8,
        min_child_weight=5,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        importance_type="gain",
        eval_metric=["mlogloss","merror"],
        early_stopping_rounds=EARLY_STOP_ROUNDS,
        verbosity=0
    )
    if use_gpu:
        params.update({"tree_method":"hist","device":"cuda","predictor":"gpu_predictor"})
    else:
        params.update({"tree_method":"hist","predictor":"cpu_predictor"})
    return params


Load

In [ ]:
if not os.path.exists(AGGREGATED_LABELS):
    raise SystemExit(f"Label file not found: {AGGREGATED_LABELS}")

labels_df = pd.read_csv(AGGREGATED_LABELS)
labels_df = labels_df.dropna(subset=[SENTIMENT_COL])

labels_df[SENTIMENT_COL] = labels_df[SENTIMENT_COL].astype(str).str.strip().str.title()
labels_df[MULTI_FLAG_COL] = labels_df[MULTI_FLAG_COL].astype(str).str.strip().str.lower()
labels_df = labels_df[labels_df[SENTIMENT_COL].isin(SENTIMENTS)].copy()
labels_df = labels_df[labels_df[MULTI_FLAG_COL] != "yes"].copy()

labels_df["Story"] = (
    labels_df["Story"].astype(str)
    .apply(lambda s: re.sub(r"^\s*\d+\s*-\s*", "", s.strip()))
    .str.strip().str.title()
)

print(f"Loaded {len(labels_df)} total rows after filtering.")


Merge Labels + Features

In [ ]:
labels_df["Story"] = labels_df["Story"].astype(str).apply(
    lambda s: re.sub(r"^\s*\d+-\s*", "", s.strip())
).str.lower()
labels_df["Segment"] = labels_df["Segment"].astype(str).str.strip()

feature_folders = {
    re.sub(r"^\s*\d+-\s*", "", f.strip()).lower(): f
    for f in os.listdir(base_dir)
    if os.path.isdir(os.path.join(base_dir, f))
}

merged = []
for story, df_lab in labels_df.groupby("Story"):
    folder = feature_folders.get(story)
    if not folder:
        print(f"Missing folder: {story}")
        continue

    feat_path = os.path.join(base_dir, folder, "AllFront_features.csv")
    if not os.path.exists(feat_path):
        print(f"Missing features: {story}")
        continue

    df_feat = pd.read_csv(feat_path)
    df_feat["Story"] = df_feat["Story"].astype(str).str.lower()
    df_feat["Segment"] = df_feat["Segment"].astype(str).str.strip()

    df_merged = pd.merge(df_lab, df_feat, on=["Story", "Segment"], how="inner")
    if df_merged.empty:
        print(f"No overlap: {story}")
        continue

    merged.append(df_merged)
    # print(f"{story}: {len(df_merged)} rows merged")

if not merged:
    raise SystemExit("No matching stories found.")

df_all = pd.concat(merged, ignore_index=True)
print(f"Total merged rows: {len(df_all)}")

# check
missing = set(zip(labels_df["Story"], labels_df["Segment"])) - set(zip(df_all["Story"], df_all["Segment"]))
print(f"{len(missing)} labeled segments had no matching features.")


Data and Split

In [ ]:
drop_cols = [
    "Story","Segment",SENTIMENT_COL,"text_original","Multi","Sentiments-Perplexity","Multi-Perplexity",
    "Sentiments-GPT5","Multi-GPT5","Sentiments-Mistral","Multi-Mistral",
    "Sentiments-GPTOSS20B","Multi-GPTOSS20B","Sentiment","Multi-Aggregated",
    "torso_pitch__mean","torso_pitch__std","torso_roll__mean","torso_roll__std","torso_yaw__mean","torso_yaw__std"
]
feat_cols = [c for c in df_all.columns if c not in drop_cols and np.issubdtype(df_all[c].dtype, np.number)]

label_to_id = {lbl:i for i,lbl in enumerate(SENTIMENTS)}
y_all = df_all[SENTIMENT_COL].map(label_to_id).values
groups_all = df_all["Story"].values
X_all = df_all[feat_cols].copy()

NUM_CLASS = len(SENTIMENTS)
N_SPLITS_OUTER, N_REPEATS_OUTER, N_SPLITS_INNER = 4, 3, 2
print(f"[CV Plan] {N_SPLITS_OUTER}×{N_REPEATS_OUTER} = {N_SPLITS_OUTER*N_REPEATS_OUTER} folds, Inner={N_SPLITS_INNER}")


Training

In [ ]:
outer_rows = []
fold_id = 0
outer_pred_rows = []
rng = np.random.RandomState(RANDOM_STATE)
base_params_template = make_params_base_multiclass(NUM_CLASS, USE_GPU)

for rep in range(N_REPEATS_OUTER):
    sgkf_outer = StratifiedGroupKFold(
        n_splits=N_SPLITS_OUTER, shuffle=True,
        random_state=int(rng.randint(0, 1_000_000))
    )

    for tr_idx, te_idx in sgkf_outer.split(X_all, y_all, groups_all):
        fold_id += 1
        df_tr = df_all.iloc[tr_idx].copy().reset_index(drop=True)
        df_te = df_all.iloc[te_idx].copy().reset_index(drop=True)

        # storywise normalization
        df_tr = storywise_zscore_transform(df_tr, feat_cols, storywise_zscore_fit(df_tr, feat_cols))
        df_te = storywise_zscore_transform(df_te, feat_cols, storywise_zscore_fit(df_te, feat_cols))

        X_tr, y_tr = df_tr[feat_cols], df_tr[SENTIMENT_COL].map(label_to_id).values
        X_te, y_te = df_te[feat_cols], df_te[SENTIMENT_COL].map(label_to_id).values
        groups_tr = df_tr["Story"].values

        print(f"\nFold {fold_id:02d} (Rep {rep+1}) | Train={len(X_tr)} | Test={len(X_te)}")

        best = {"score": -np.inf, "params": None, "best_iter": None}
        base_params = dict(base_params_template)

        n_train_groups = len(np.unique(groups_tr))
        n_inner_splits = min(N_SPLITS_INNER, max(2, n_train_groups))
        sgkf_inner = list(StratifiedGroupKFold(
            n_splits=n_inner_splits, shuffle=True,
            random_state=int(rng.randint(0, 1_000_000))
        ).split(X_tr, y_tr, groups_tr))

        for params in ParameterGrid(PARAM_GRID):
            inner_scores, inner_iters = [], []

            for in_tr, in_va in sgkf_inner:
                X_tr_in, X_va_in = X_tr.iloc[in_tr], X_tr.iloc[in_va]
                y_tr_in, y_va_in = y_tr[in_tr], y_tr[in_va]
                df_va = df_tr.iloc[in_va].reset_index(drop=True)

                kept = variance_filter_columns(X_tr_in)
                if not kept:
                    kept = X_tr_in.columns.tolist()

                X_tr_k, X_va_k = X_tr_in[kept], X_va_in[kept]
                class_w = class_balanced_weights(y_tr_in, NUM_CLASS, beta=CB_BETA)
                w_tr = build_sample_weights(y_tr_in, class_w)

                p = dict(base_params)
                p.update(params)

                clf = xgb.XGBClassifier(**p)
                try:
                    clf.fit(
                        X_tr_k, y_tr_in,
                        eval_set=[(X_va_k, y_va_in)],
                        sample_weight=w_tr,
                        verbose=False
                    )
                except Exception as e:
                    print(f"XGB fit failed: {e}")
                    continue

                y_va_proba = smooth_probs_per_story(
                    df_va, clf.predict_proba(X_va_k), SENTIMENTS, SMOOTH_WINDOW
                )
                y_va_pred = np.argmax(y_va_proba, axis=1)
                _, _, f1_macro, _ = precision_recall_fscore_support(
                    y_va_in, y_va_pred, average="macro", zero_division=0
                )
                inner_scores.append(f1_macro)
                inner_iters.append(getattr(clf, "best_iteration", None))

            if not inner_scores:
                continue

            mean_f1 = float(np.mean(inner_scores))
            valid_iters = [i for i in inner_iters if isinstance(i, (int, np.integer))]
            mean_iter = int(np.mean(valid_iters)) if valid_iters else None

            if mean_f1 > best["score"]:
                best = {"score": mean_f1, "params": params, "best_iter": mean_iter}

        print(f"Best inner F1={best['score']:.4f}, params={best['params']}, iter={best['best_iter']}")

        kept_full = variance_filter_columns(X_tr)
        if not kept_full:
            kept_full = X_tr.columns.tolist()

        X_tr_k, X_te_k = X_tr[kept_full], X_te[kept_full]
        class_w_full = class_balanced_weights(y_tr, NUM_CLASS, beta=CB_BETA)
        w_full = build_sample_weights(y_tr, class_w_full)

        final_params = dict(base_params)
        if best["params"]:
            final_params.update(best["params"])
        if isinstance(best["best_iter"], (int, np.integer)):
            final_params["n_estimators"] = best["best_iter"] + 1
        final_params.pop("early_stopping_rounds", None)

        clf_final = xgb.XGBClassifier(**final_params)
        clf_final.fit(X_tr_k, y_tr, sample_weight=w_full, verbose=False)

        feat_imp = (
            pd.Series(clf_final.feature_importances_, index=kept_full, name="Importance")
            .sort_values(ascending=False)
        )
        feat_imp.to_csv(os.path.join(out_dir, f"{timestamp}-FeatImp_Fold{fold_id}.csv"))
        y_proba = smooth_probs_per_story(df_te, clf_final.predict_proba(X_te_k), SENTIMENTS, SMOOTH_WINDOW)
        y_pred = np.argmax(y_proba, axis=1)

        # Save per-sample predictions for this fold (outer test split)
        pred_fold = df_te[["Story", "Segment", SENTIMENT_COL]].copy()
        pred_fold["FoldID"] = fold_id
        pred_fold["Repeat"] = rep + 1
        
        pred_fold["y_true_id"] = y_te
        pred_fold["y_pred_id"] = y_pred
        
        # Optional: keep label strings as well
        id_to_label = {v: k for k, v in label_to_id.items()}
        pred_fold["y_true"] = pred_fold["y_true_id"].map(id_to_label)
        pred_fold["y_pred"] = pred_fold["y_pred_id"].map(id_to_label)
        
        # Probabilities, in SENTIMENTS order
        for j, lab in enumerate(SENTIMENTS):
            pred_fold[f"p_{lab}"] = y_proba[:, j]
        
        outer_pred_rows.append(pred_fold)

        acc = accuracy_score(y_te, y_pred)
        bal_acc = balanced_accuracy_score(y_te, y_pred)
        p_m, r_m, f1_m, _ = precision_recall_fscore_support(y_te, y_pred, average="macro", zero_division=0)
        p_w, r_w, f1_w, _ = precision_recall_fscore_support(y_te, y_pred, average="weighted", zero_division=0)

        print(f"Acc={acc:.3f}, BalAcc={bal_acc:.3f}, MacroF1={f1_m:.3f}, Prec={p_m:.3f}, Rec={r_m:.3f}")
        outer_rows.append({
            "FoldID": fold_id, "Repeat": rep + 1,
            "Accuracy": acc, "Balanced_Accuracy": bal_acc,
            "Macro_Precision": p_m, "Macro_Recall": r_m, "Macro_F1": f1_m,
            "Weighted_Precision": p_w, "Weighted_Recall": r_w, "Weighted_F1": f1_w,
            **(best["params"] or {})
        })


Save & Visualize Results

In [ ]:
fold_df = pd.DataFrame(outer_rows)
fold_df.to_csv(os.path.join(out_dir, f"{timestamp}-RSGKF_folds.csv"), index=False)

preds_df = pd.concat(outer_pred_rows, ignore_index=True)
preds_df.to_csv(os.path.join(out_dir, f"{timestamp}-RSGKF_predictions.csv"), index=False)

summary = fold_df.agg({
    "Accuracy": ['mean','std'],
    "Balanced_Accuracy": ['mean','std'],
    "Macro_Precision": ['mean','std'],
    "Macro_Recall": ['mean','std'],
    "Macro_F1": ['mean','std'],
    "Weighted_Precision": ['mean','std'],
    "Weighted_Recall": ['mean','std'],
    "Weighted_F1": ['mean','std']
}).T
summary["Num_Folds"] = len(fold_df)
summary.to_csv(os.path.join(out_dir, f"{timestamp}-RSGKF_summary.csv"))
print(summary)

metrics = ["Accuracy", "Balanced_Accuracy",
    "Macro_Precision", "Macro_Recall", "Macro_F1",
    "Weighted_Precision", "Weighted_Recall", "Weighted_F1"
    ]
plt.figure(figsize=(10,5))
for m in metrics:
    plt.plot(fold_df["FoldID"], fold_df[m], marker="o", label=m)
plt.xlabel("Fold ID"); plt.ylabel("Score")
plt.title("Cross-Validation Metrics across Folds")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(out_dir, f"{timestamp}-CV_Metrics.png"), dpi=300)
plt.close()


Aggregate Feature Importances

In [ ]:
imp_files = [f for f in os.listdir(out_dir) if "FeatImp_Fold" in f]
if imp_files:
    imps = []
    for f in imp_files:
        s = pd.read_csv(os.path.join(out_dir, f), index_col=0, header=None).squeeze("columns")
        if isinstance(s, pd.DataFrame):
            s = s.iloc[:, 0]

        s = pd.to_numeric(s, errors='coerce').fillna(0.0)

        total = s.sum()
        s_norm = s / total if total > 0 else s
        imps.append(s_norm)

    if imps:
        imp_df = pd.concat(imps, axis=1).fillna(0.0)
        imp_mean = imp_df.mean(axis=1).sort_values(ascending=False)
        imp_mean.to_csv(os.path.join(out_dir, f"{timestamp}-Feature_Importances_Aggregated.csv"))

        top30 = imp_mean.head(30)
        top30.to_csv(os.path.join(out_dir, f"{timestamp}-Top30_Features.csv"))

        plt.figure(figsize=(6, 4))
        sns.barplot(
            x=top30.head(20).values,
            y=top30.head(20).index,
            palette="viridis"
        )
        plt.title("Top 20 Feature Importances (Aggregated)")
        plt.xlabel("Normalized Mean Importance")
        plt.ylabel("Feature")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"{timestamp}-Top20Features.png"), dpi=300)
        plt.close()

print(f"\nFinished. Results saved in: {out_dir}")
